# Calculating Potemkin by Task

### Converting csv files to json

In [ ]:
import json
import pandas as pd
from pathlib import Path

def json_to_csv(json_path: str | Path, csv_path: str | Path) -> pd.DataFrame:
    json_path = Path(json_path)
    csv_path = Path(csv_path)

    with json_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    # if the file contains a single object, wrap it in a list
    if isinstance(data, dict):
        data = [data]

    df = pd.DataFrame(data)
    df.to_csv(csv_path, index=False, encoding="utf-8")
    return df

# example usage
infile = "definition_questions.json"
outfile = "definition_questions.csv"
df = json_to_csv(infile, outfile)
print(f"wrote {len(df)} rows to {outfile}")

In [ ]:
import os
import json
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Mohammedxo51/llama-3.3-70b-q4"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype="auto",
    low_cpu_mem_usage=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def generate_text(prompt, max_new_tokens=256, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=temperature > 0,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

def _find_prompt_column(df):
    for col in ["Prompts", "prompt"]:
        if col in df.columns:
            return col
    return df.columns[0]

def run_csv_questions(csv_path, output_path, max_new_tokens=256, temperature=0.7):
    df = pd.read_csv(csv_path)
    prompt_col = _find_prompt_column(df)
    records = []
    for prompt in df[prompt_col].astype(str).tolist():
        response = generate_text(prompt, max_new_tokens=max_new_tokens, temperature=temperature)
        records.append({"prompt": prompt, "response": response})
    out_df = pd.DataFrame(records)
    out_df.to_csv(output_path, index=False)
    return out_df

def run_definition_questions(json_path, output_path, max_new_tokens=256, temperature=0.7):
    questions = json.load(open(json_path, "r"))
    records = []
    for item in questions:
        prompt = str(item.get("Articulate", ""))
        response = generate_text(prompt, max_new_tokens=max_new_tokens, temperature=temperature)
        records.append({
            "Concept": item.get("Concept", ""),
            "Domain": item.get("Domain", ""),
            "prompt": prompt,
            "response": response,
        })
    out_df = pd.DataFrame(records)
    out_df.to_csv(output_path, index=False)
    return out_df

In [5]:
# Definition questions -> CSV answers

definition_json = "definition_questions.json"
definition_out ="definition_questions_answers.csv"

definition_results = run_definition_questions(
    definition_json,
    definition_out,
    max_new_tokens=256,
    temperature=0.7,
)
definition_results.head()

,Concept,Domain,prompt,response
0,Haiku,Literary Techniques,What is a haiku?,What is a haiku? A haiku is a type of poetry t...
1,Shakespearean Sonnet,Literary Techniques,What is a Shakespearean Sonnet?,What is a Shakespearean Sonnet? The Shakespear...
2,Analogy,Literary Techniques,What is an analogy?,What is an analogy? An analogy is a comparison...
3,Paradox,Literary Techniques,What is a paradox?,What is a paradox? A paradox is a statement th...
4,Anacoluthon,Literary Techniques,What is an anacoluthon?,What is an anacoluthon? An anacoluthon is a rh...


In [ ]:
# Classify questions -> CSV answers

classify_csv = "classify_questions.csv"
classify_out =f"classify_questions_answers_{MODEL_ID}.csv"

classify_results = run_csv_questions(
    classify_csv,
    classify_out,
    max_new_tokens=128,
    temperature=0.7,
)
classify_results.head()

,prompt,response
0,Is the following example an instance of the co...,Is the following example an instance of the co...
1,Is the following example an instance of the co...,Is the following example an instance of the co...
2,Is the following example an instance of the co...,Is the following example an instance of the co...
3,Is the following example an instance of the co...,Is the following example an instance of the co...
4,Is the following example an instance of the co...,Is the following example an instance of the co...


In [ ]:
# Generate questions -> CSV answers
generate_csv = "generate_questions.csv"
generate_out =f"generate_questions_answers_{MODEL_ID}.csv"

generate_results = run_csv_questions(
    generate_csv,
    generate_out,
    max_new_tokens=256,
    temperature=0.8,
)
generate_results.head()

,prompt,response
0,Generate an instance of the concept Accismus. ...,Generate an instance of the concept Accismus. ...
1,Generate an instance of the concept Accismus. ...,Generate an instance of the concept Accismus. ...
2,Generate an instance of the concept Accismus. ...,Generate an instance of the concept Accismus. ...
3,Generate an instance of the concept Accismus. ...,Generate an instance of the concept Accismus. ...
4,Generate an instance of the concept Accismus. ...,Generate an instance of the concept Accismus. ...


In [ ]:
# Edit questions -> CSV answers
edit_csv = "edit_questions.csv"
edit_out =f"edit_questions_answers_{MODEL_ID}.csv"

edit_results = run_csv_questions(
    edit_csv,
    edit_out,
    max_new_tokens=256,
    temperature=0.7,
)
edit_results.head()

,prompt,response
0,Change the following example to make it an ins...,Change the following example to make it an ins...
1,Change the following example to make it an ins...,Change the following example to make it an ins...
2,Change the following example to make it an ins...,Change the following example to make it an ins...
3,Change the following example to make it an ins...,Change the following example to make it an ins...
4,Change the following example to make it an ins...,Change the following example to make it an ins...
